In [1]:
import json
import os
import pandas as pd

folder_path = r"C:\Users\dell\Desktop\Projects using DS\ipl_json"

all_balls = []

for file in os.listdir(folder_path):
    if file.endswith(".json"):
        with open(os.path.join(folder_path, file), 'r') as f:
            data = json.load(f)
            
            match_id = file.replace('.json', '')
            innings_list = data.get('innings', [])
            
            for innings in innings_list:
                team = innings.get('team', '')
                overs = innings.get('overs', [])
                
                for over in overs:
                    over_num = over.get('over', 0)
                    deliveries = over.get('deliveries', [])
                    
                    for delivery in deliveries:
                        # Wicket info
                        wicket = 0
                        player_out = ''
                        if 'wickets' in delivery:
                            wicket = 1
                            player_out = delivery['wickets'][0].get('player_out', '')
                        
                        ball = {
                            'match_id': match_id,
                            'batting_team': team,
                            'over': over_num,
                            'batter': delivery.get('batter', ''),
                            'bowler': delivery.get('bowler', ''),
                            'runs_batter': delivery['runs'].get('batter', 0),
                            'runs_extras': delivery['runs'].get('extras', 0),
                            'runs_total': delivery['runs'].get('total', 0),
                            'wicket': wicket,
                            'player_out': player_out
                        }
                        all_balls.append(ball)

balls_df = pd.DataFrame(all_balls)
print(f"Total balls: {len(balls_df)}")
balls_df.head(10)


Total balls: 295732


,match_id,batting_team,over,batter,bowler,runs_batter,runs_extras,runs_total,wicket,player_out
0,1082591,Sunrisers Hyderabad,0,DA Warner,TS Mills,0,0,0,0,
1,1082591,Sunrisers Hyderabad,0,DA Warner,TS Mills,0,0,0,0,
2,1082591,Sunrisers Hyderabad,0,DA Warner,TS Mills,4,0,4,0,
3,1082591,Sunrisers Hyderabad,0,DA Warner,TS Mills,0,0,0,0,
4,1082591,Sunrisers Hyderabad,0,DA Warner,TS Mills,0,2,2,0,
5,1082591,Sunrisers Hyderabad,0,S Dhawan,TS Mills,0,0,0,0,
6,1082591,Sunrisers Hyderabad,0,S Dhawan,TS Mills,0,1,1,0,
7,1082591,Sunrisers Hyderabad,1,S Dhawan,A Choudhary,1,0,1,0,
8,1082591,Sunrisers Hyderabad,1,DA Warner,A Choudhary,4,0,4,0,
9,1082591,Sunrisers Hyderabad,1,DA Warner,A Choudhary,0,1,1,0,


In [3]:
# Batting Stats
batting_stats = balls_df.groupby('batter').agg(
    total_runs=('runs_batter', 'sum'),
    total_balls=('runs_batter', 'count'),
).reset_index()

# Dismissals — player_out se nikalo
dismissals = balls_df[balls_df['player_out'] == balls_df['batter']].groupby('player_out').size().reset_index()
dismissals.columns = ['batter', 'dismissals']

# Merge karo
batting_stats = batting_stats.merge(dismissals, on='batter', how='left')
batting_stats['dismissals'] = batting_stats['dismissals'].fillna(0).astype(int)

# Strike Rate aur Average
batting_stats['strike_rate'] = (batting_stats['total_runs'] / batting_stats['total_balls'] * 100).round(2)
batting_stats['average'] = (batting_stats['total_runs'] / batting_stats['dismissals'].replace(0, 1)).round(2)

# Top 10 batters
print(batting_stats.sort_values('total_runs', ascending=False).head(10).to_string(index=False))

        batter  total_runs  total_balls  dismissals  strike_rate  average
       V Kohli        9346         7127         229       131.14    40.81
     RG Sharma        7331         5659         244       129.55    30.05
      S Dhawan        6769         5483         187       123.45    36.20
     DA Warner        6567         4849         158       135.43    41.56
      KL Rahul        5828         4302         125       135.47    46.62
      SK Raina        5536         4177         162       132.54    34.17
      MS Dhoni        5439         4101         139       132.63    39.13
     AM Rahane        5367         4391         172       122.23    31.20
AB de Villiers        5181         3487         122       148.58    42.47
     SV Samson        5181         3770         159       137.43    32.58


In [5]:
# Bowling Stats
bowling_stats = balls_df.groupby('bowler').agg(
    total_balls=('runs_total', 'count'),
    total_runs=('runs_total', 'sum'),
    total_wickets=('wicket', 'sum')
).reset_index()

# Overs calculate karo
bowling_stats['overs'] = (bowling_stats['total_balls'] / 6).round(1)

# Economy Rate
bowling_stats['economy'] = (bowling_stats['total_runs'] / bowling_stats['overs']).round(2)

# Bowling Average
bowling_stats['bowling_avg'] = (bowling_stats['total_runs'] / bowling_stats['total_wickets'].replace(0, 1)).round(2)

# Top 10 wicket takers
print(bowling_stats.sort_values('total_wickets', ascending=False).head(10).to_string(index=False))

     bowler  total_balls  total_runs  total_wickets  overs  economy  bowling_avg
    B Kumar         4769        6045            243  794.8     7.61        24.88
  YS Chahal         4167        5501            242  694.5     7.92        22.73
  SP Narine         4730        5379            229  788.3     6.82        23.49
  JJ Bumrah         3792        4582            208  632.0     7.25        22.03
   DJ Bravo         3296        4436            207  549.3     8.08        21.43
   R Ashwin         4868        5721            205  811.3     7.05        27.91
  PP Chawla         3895        5179            201  649.2     7.98        25.77
  RA Jadeja         4329        5525            189  721.5     7.66        29.23
 SL Malinga         2974        3486            188  495.7     7.03        18.54
Rashid Khan         3585        4390            187  597.5     7.35        23.48


In [6]:
batting_stats.to_csv(r"C:\Users\dell\Desktop\Projects using DS\ipl_json\batting_stats.csv", index=False)
bowling_stats.to_csv(r"C:\Users\dell\Desktop\Projects using DS\ipl_json\bowling_stats.csv", index=False)
print("Saved!")

Saved!
